# Week 1 - ML Fundamentals + Data Preprocessing
## Assignments & Mini Project 1: Titanic Survival Prediction

---
### Overview
This notebook implements all tasks requested for Week 1:
1. **Assignment 1**: Load a dataset using Pandas and summarize basic stats (`.info()`, `.describe()`).
2. **Assignment 2**: Handle missing data using mean/median imputation.
3. **Assignment 3**: Encode categorical variables using `LabelEncoder` and `OneHotEncoder`.
4. **Mini Project 1**: Clean missing data, encode `Sex` and `Embarked`, visualize age distribution with Matplotlib/Seaborn, and export `titanic_cleaned.csv`.

## 1. Assignment 1: Load Dataset & Summarize Basic Stats

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 1. Load dataset
df = pd.read_csv('Titanic-Dataset.csv')
print(f'Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# Dataset information (.info())
df.info()

In [ ]:
# Numerical summary (.describe())
df.describe()

In [ ]:
# Categorical summary
df.describe(include=['object'])

In [ ]:
# Null counts and percentages
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100
null_summary = pd.DataFrame({'Missing Values': null_counts, 'Percentage (%)': null_pct.round(2)})
null_summary[null_summary['Missing Values'] > 0]

## 2. Assignment 2: Handle Missing Data (Mean / Median Imputation)

In [ ]:
# Mean imputation with Pandas fillna
df_mean_demo = df.copy()
mean_val = df_mean_demo['Age'].mean()
df_mean_demo['Age_Mean'] = df_mean_demo['Age'].fillna(mean_val)
print(f'Calculated Mean: {mean_val:.2f}')
print(f'Missing after mean imputation: {df_mean_demo["Age_Mean"].isnull().sum()}')

In [ ]:
# Median imputation with Scikit-Learn SimpleImputer
imputer = SimpleImputer(strategy='median')
df_median_demo = df.copy()
df_median_demo['Age_Median'] = imputer.fit_transform(df[['Age']])
print(f'Calculated Median: {imputer.statistics_[0]:.2f}')
print(f'Missing after median imputation: {df_median_demo["Age_Median"].isnull().sum()}')

## 3. Assignment 3: Encode Categorical Variables (LabelEncoder & OneHotEncoder)

In [ ]:
# LabelEncoder on 'Sex'
le = LabelEncoder()
df_le = df[['Sex']].copy()
df_le['Sex_Encoded'] = le.fit_transform(df_le['Sex'])
print('LabelEncoder Classes:', dict(zip(le.classes_, le.transform(le.classes_))))
df_le.drop_duplicates()

In [ ]:
# OneHotEncoder on 'Embarked'
ohe = OneHotEncoder(sparse_output=False, dtype=int)
embarked_clean = df[['Embarked']].fillna('S')
embarked_encoded = ohe.fit_transform(embarked_clean)
encoded_columns = [f'Embarked_{cat}' for cat in ohe.categories_[0]]
df_ohe = pd.DataFrame(embarked_encoded, columns=encoded_columns)
pd.concat([embarked_clean, df_ohe], axis=1).drop_duplicates()

## 4. Mini Project 1: Titanic Data Cleaning Project

In [ ]:
# 4.1 Clean Missing Data
df_clean = df.copy()
raw_age = df_clean['Age'].copy()

# Impute Age with median
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())

# Impute Embarked with mode
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])

# Drop high missingness Cabin column
df_clean.drop(columns=['Cabin'], inplace=True)

# 4.2 Encode Sex & Embarked
df_clean['Sex_Code'] = le.fit_transform(df_clean['Sex'])
embarked_dummies = pd.get_dummies(df_clean['Embarked'], prefix='Embarked', dtype=int)
df_clean = pd.concat([df_clean, embarked_dummies], axis=1)

print('Null values remaining in processed subset:')
print(df_clean[['Age', 'Embarked', 'Sex_Code', 'Fare']].isnull().sum())

In [ ]:
# 4.3 Visualize Age Distribution with Seaborn & Matplotlib
sns.set_theme(style='whitegrid', palette='muted')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Titanic Dataset - Age Distribution Analysis', fontsize=16, fontweight='bold', y=0.98)

# Raw vs Imputed Age
sns.histplot(raw_age.dropna(), color='#3498db', kde=True, stat='density', label='Raw Age', ax=axes[0, 0], alpha=0.4, bins=30)
sns.histplot(df_clean['Age'], color='#e74c3c', kde=True, stat='density', label='Median Imputed Age', ax=axes[0, 0], alpha=0.3, bins=30)
axes[0, 0].set_title('Age Distribution: Raw vs. Median Imputed', fontweight='bold')
axes[0, 0].legend()

# KDE by Survival
sns.kdeplot(data=df_clean[df_clean['Survived'] == 0], x='Age', label='Did not survive (0)', color='#e74c3c', fill=True, alpha=0.3, ax=axes[0, 1])
sns.kdeplot(data=df_clean[df_clean['Survived'] == 1], x='Age', label='Survived (1)', color='#2ecc71', fill=True, alpha=0.3, ax=axes[0, 1])
axes[0, 1].set_title('Age KDE by Survival Status', fontweight='bold')
axes[0, 1].legend()

# Boxplot by Pclass
sns.boxplot(data=df_clean, x='Pclass', y='Age', hue='Pclass', palette='Set2', legend=False, ax=axes[1, 0])
axes[1, 0].set_title('Age Distribution across Passenger Classes', fontweight='bold')

# Violin by Sex and Survival
sns.violinplot(data=df_clean, x='Sex', y='Age', hue='Survived', split=True, palette={0: '#e74c3c', 1: '#2ecc71'}, ax=axes[1, 1])
axes[1, 1].set_title('Age Distribution by Gender & Survival', fontweight='bold')

plt.tight_layout()
plt.savefig('age_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4.4 Output Cleaned Dataset as new CSV
features = ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Sex_Code', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Embarked_C', 'Embarked_Q', 'Embarked_S']
df_final = df_clean[features]
df_final.to_csv('titanic_cleaned.csv', index=False)
print(f'Cleaned dataset saved as titanic_cleaned.csv (Shape: {df_final.shape})')
df_final.head()